---
## 1️⃣ Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Core imports
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D

# TriMedAgent Orchestrator
from src import TriMedOrchestrator, PipelineResult

print("✓ Imports successful")
print(f"  Project Root: {PROJECT_ROOT}")

---
## 2️⃣ Initialize Orchestrator

In [ ]:
# Initialize Orchestrator with default settings
orchestrator = TriMedOrchestrator(
    timeout=120,  # 2 minutes timeout per request
    conv_template="llava_v1"  # Conversation template for LLaVA
)

print("✓ Orchestrator initialized")
print(f"\nWorker URLs:")
for name, url in orchestrator.worker_urls.items():
    print(f"  • {name}: {url}")

In [ ]:
# Health Check - Verify workers are online
print("🔍 Checking worker connectivity...\n")

health = orchestrator.health_check()

for worker, is_online in health.items():
    status = "✅ Online" if is_online else "❌ Offline"
    print(f"  {worker}: {status}")

all_online = all(health.values())
if all_online:
    print("\n🎉 All workers are ready!")
else:
    print("\n⚠️ Some workers are offline. Pipeline may fail.")

---
## 3️⃣ Load Test Image

In [ ]:
# Load sample image
# Option 1: From local file
image_path = PROJECT_ROOT / "images" / "sample_xray.jpg"

# Option 2: Create placeholder if no sample exists
if not image_path.exists():
    print(f"⚠️ Sample image not found at: {image_path}")
    print("  Please provide an image path:")
    # Uncomment and set your image path:
    # image_path = Path("your/image/path.jpg")
    
    # Or download a sample:
    # !wget -O images/sample_xray.jpg "YOUR_URL"
else:
    image = Image.open(image_path)
    print(f"✓ Image loaded: {image_path.name}")
    print(f"  Size: {image.size}")
    print(f"  Mode: {image.mode}")
    
    # Display image
    plt.figure(figsize=(8, 8))
    plt.imshow(image, cmap='gray' if image.mode == 'L' else None)
    plt.title("Input Medical Image")
    plt.axis('off')
    plt.show()

---
## 4️⃣ Run Individual Stages (Optional)

You can run each stage independently for debugging or analysis.

In [ ]:
# Stage 1: Triage Only
if 'image' in dir():
    print("🔬 Running Triage (Stage 1)...")
    
    try:
        triage_result = orchestrator.triage(image)
        
        print(f"\n📊 Triage Result:")
        print(f"  Modality: {triage_result.modality}")
        print(f"  Confidence: {triage_result.confidence:.1%}")
        
        if triage_result.all_scores:
            print(f"\n  All Scores:")
            sorted_scores = sorted(
                triage_result.all_scores.items(), 
                key=lambda x: x[1], 
                reverse=True
            )
            for label, score in sorted_scores[:5]:
                bar = "█" * int(score * 20)
                print(f"    {label}: {score:.1%} {bar}")
                
    except Exception as e:
        print(f"❌ Triage failed: {e}")

In [ ]:
# Stage 2: Ask LLaVA (without detection)
if 'image' in dir():
    print("🤖 Asking LLaVA-Med...")
    
    question = "What abnormalities can you observe in this image?"
    
    try:
        # Build context from triage if available
        context = ""
        if 'triage_result' in dir():
            context = f"[System: This is a {triage_result.modality} image]"
        
        response = orchestrator.ask_llava(image, question, context)
        
        print(f"\n❓ Question: {question}")
        print(f"\n💬 LLaVA Response:")
        print("-" * 50)
        print(response)
        print("-" * 50)
        
    except Exception as e:
        print(f"❌ LLaVA failed: {e}")

---
## 5️⃣ Run Full Pipeline 🚀

Execute the complete **Perceive → Reason → Verify → Act** chain.

In [ ]:
# Define user query (this triggers detection if action keywords present)
user_query = "Find and segment any tumors or abnormalities in this image"

print(f"🎯 Query: '{user_query}'")
print("=" * 60)

In [ ]:
# Run the full pipeline
if 'image' in dir():
    print("🚀 Running Full Pipeline...\n")
    
    try:
        result: PipelineResult = orchestrator.run_full_chain(
            image=image,
            user_query=user_query,
            skip_gatekeeper=False,
            skip_segmentation=False
        )
        
        print("\n" + "=" * 60)
        print("📋 PIPELINE SUMMARY")
        print("=" * 60)
        
        # Stage 1 Results
        print(f"\n🔬 Stage 1 - Triage:")
        if result.triage:
            print(f"   Modality: {result.triage.modality}")
            print(f"   Confidence: {result.triage.confidence:.1%}")
        
        # Stage 2 Results
        print(f"\n🤖 Stage 2 - LLaVA Reasoning:")
        print(f"   Context: {result.context_injected[:80]}...")
        print(f"   Response: {result.llava_response[:200]}...")
        
        # Stage 3 Results
        print(f"\n🔍 Stage 3 - Detection (DINO):")
        print(f"   Raw Boxes Found: {len(result.dino_raw_boxes)}")
        
        # Stage 4 Results
        print(f"\n✅ Stage 4 - Gatekeeper Verification:")
        print(f"   Verified Boxes: {len(result.verified_boxes)}")
        print(f"   Rejected Boxes: {len(result.rejected_boxes)}")
        
        # Stage 5 Results
        print(f"\n🎭 Stage 5 - Segmentation (MedSAM):")
        print(f"   Masks Generated: {len(result.masks)}")
        
        # Metadata
        print(f"\n⏱️ Execution Time: {result.execution_time:.2f}s")
        print(f"   Stopped At: {result.stopped_at_stage}")
        print(f"   Complete: {result.pipeline_complete}")
        
        if result.errors:
            print(f"\n⚠️ Errors:")
            for error in result.errors:
                print(f"   • {error}")
                
    except Exception as e:
        print(f"\n❌ Pipeline failed: {e}")
        import traceback
        traceback.print_exc()

---
## 6️⃣ Visualization 📊

**Color Legend:**
- 🔴 **Red Dashed**: Raw DINO detections (before verification)
- 🟢 **Green Solid**: Verified boxes (after Gatekeeper)
- 🔵 **Blue Overlay**: MedSAM segmentation masks (α=0.5)

In [ ]:
def visualize_pipeline_result(
    image: Image.Image,
    result: PipelineResult,
    figsize: tuple = (16, 8),
    save_path: str = None
):
    """
    Visualize pipeline results with clear annotations.
    
    - Raw DINO boxes: Red dashed lines
    - Verified boxes: Green solid lines
    - Masks: Blue semi-transparent overlay
    """
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Convert image to RGB if needed
    if image.mode == 'L':
        display_img = image.convert('RGB')
    else:
        display_img = image.copy()
    
    img_array = np.array(display_img)
    
    # ===== Left Panel: Raw Detections =====
    ax1 = axes[0]
    ax1.imshow(img_array)
    ax1.set_title("Raw DINO Detections", fontsize=14, fontweight='bold')
    ax1.axis('off')
    
    # Draw raw boxes (red dashed)
    for i, box in enumerate(result.dino_raw_boxes):
        x1, y1, x2, y2 = box
        width = x2 - x1
        height = y2 - y1
        
        rect = patches.Rectangle(
            (x1, y1), width, height,
            linewidth=2,
            edgecolor='red',
            facecolor='none',
            linestyle='--'
        )
        ax1.add_patch(rect)
        
        # Add box number
        ax1.text(
            x1, y1 - 5, f"Box {i+1}",
            color='red',
            fontsize=10,
            fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
        )
    
    ax1.text(
        0.02, 0.98, f"Total: {len(result.dino_raw_boxes)} boxes",
        transform=ax1.transAxes,
        fontsize=12,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
    )
    
    # ===== Right Panel: Verified Results =====
    ax2 = axes[1]
    ax2.imshow(img_array)
    ax2.set_title("Verified Results (After Gatekeeper)", fontsize=14, fontweight='bold')
    ax2.axis('off')
    
    # Draw raw boxes (faded red) for comparison
    for box in result.dino_raw_boxes:
        x1, y1, x2, y2 = box
        width = x2 - x1
        height = y2 - y1
        
        rect = patches.Rectangle(
            (x1, y1), width, height,
            linewidth=1,
            edgecolor='red',
            facecolor='none',
            linestyle='--',
            alpha=0.3
        )
        ax2.add_patch(rect)
    
    # Draw verified boxes (green solid)
    for i, box in enumerate(result.verified_boxes):
        x1, y1, x2, y2 = box
        width = x2 - x1
        height = y2 - y1
        
        rect = patches.Rectangle(
            (x1, y1), width, height,
            linewidth=3,
            edgecolor='limegreen',
            facecolor='none',
            linestyle='-'
        )
        ax2.add_patch(rect)
        
        # Add verified label
        ax2.text(
            x1, y1 - 5, f"✓ Verified {i+1}",
            color='green',
            fontsize=10,
            fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
        )
    
    # Overlay masks (blue, semi-transparent)
    if result.masks:
        try:
            # Create mask overlay
            mask_overlay = np.zeros((*img_array.shape[:2], 4), dtype=np.float32)
            
            for mask in result.masks:
                # Handle different mask formats
                if isinstance(mask, str):
                    # Base64 encoded mask
                    import base64
                    from io import BytesIO
                    mask_bytes = base64.b64decode(mask)
                    mask_img = Image.open(BytesIO(mask_bytes))
                    mask_array = np.array(mask_img)
                elif isinstance(mask, np.ndarray):
                    mask_array = mask
                else:
                    continue
                
                # Resize mask to image size if needed
                if mask_array.shape[:2] != img_array.shape[:2]:
                    mask_pil = Image.fromarray(mask_array.astype(np.uint8))
                    mask_pil = mask_pil.resize(display_img.size, Image.NEAREST)
                    mask_array = np.array(mask_pil)
                
                # Apply blue color with transparency
                mask_bool = mask_array > 0
                if len(mask_bool.shape) > 2:
                    mask_bool = mask_bool[:, :, 0]
                
                mask_overlay[mask_bool, 0] = 0.0    # R
                mask_overlay[mask_bool, 1] = 0.5    # G
                mask_overlay[mask_bool, 2] = 1.0    # B
                mask_overlay[mask_bool, 3] = 0.5    # Alpha
            
            ax2.imshow(mask_overlay)
            
        except Exception as e:
            print(f"Warning: Could not overlay masks: {e}")
    
    # Stats text
    stats = f"Verified: {len(result.verified_boxes)}/{len(result.dino_raw_boxes)} boxes"
    if result.masks:
        stats += f"\nMasks: {len(result.masks)}"
    
    ax2.text(
        0.02, 0.98, stats,
        transform=ax2.transAxes,
        fontsize=12,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
    )
    
    # Add legend
    legend_elements = [
        Line2D([0], [0], color='red', linestyle='--', linewidth=2, label='Raw DINO Detection'),
        Line2D([0], [0], color='limegreen', linestyle='-', linewidth=3, label='Verified (Gatekeeper)'),
        patches.Patch(facecolor='royalblue', alpha=0.5, label='MedSAM Mask')
    ]
    fig.legend(
        handles=legend_elements,
        loc='lower center',
        ncol=3,
        fontsize=11,
        frameon=True,
        bbox_to_anchor=(0.5, -0.02)
    )
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"💾 Figure saved to: {save_path}")
    
    plt.show()
    
    return fig

In [ ]:
# Visualize results
if 'result' in dir() and 'image' in dir():
    print("📊 Generating Visualization...\n")
    
    fig = visualize_pipeline_result(
        image=image,
        result=result,
        figsize=(16, 8),
        save_path=None  # Set path to save: "output/result.png"
    )
else:
    print("⚠️ No results to visualize. Run the pipeline first.")

---
## 7️⃣ Detailed Gatekeeper Analysis

Examine which boxes were accepted/rejected and why.

In [ ]:
# Gatekeeper Analysis
if 'result' in dir() and result.gatekeeper_results:
    print("🔍 Gatekeeper Decision Analysis")
    print("=" * 60)
    
    for i, gk in enumerate(result.gatekeeper_results):
        status = "✅ VERIFIED" if gk.is_valid else "❌ REJECTED"
        
        print(f"\nBox {i+1}: {status}")
        print(f"  Coordinates: [{gk.box[0]:.0f}, {gk.box[1]:.0f}, {gk.box[2]:.0f}, {gk.box[3]:.0f}]")
        print(f"  Pathology Score: {gk.pathology_score:.1%}")
        print(f"  Normal Score: {gk.normal_score:.1%}")
        print(f"  Reason: {gk.reason}")
        
        # Visual bar
        path_bar = "█" * int(gk.pathology_score * 20)
        norm_bar = "░" * int(gk.normal_score * 20)
        print(f"  [Pathology] {path_bar}{norm_bar} [Normal]")
    
    print("\n" + "=" * 60)
    print(f"Summary: {len(result.verified_boxes)} verified, {len(result.rejected_boxes)} rejected")
    
else:
    print("No Gatekeeper results available.")

---
## 8️⃣ Export Results

In [ ]:
import json
from datetime import datetime

def export_result_to_json(result: PipelineResult, output_path: str):
    """Export pipeline result to JSON."""
    
    export_data = {
        "timestamp": datetime.now().isoformat(),
        "query": user_query if 'user_query' in dir() else "",
        "execution_time_seconds": result.execution_time,
        "pipeline_complete": result.pipeline_complete,
        "stopped_at_stage": result.stopped_at_stage,
        
        "triage": {
            "modality": result.triage.modality if result.triage else None,
            "confidence": result.triage.confidence if result.triage else None,
        } if result.triage else None,
        
        "llava_response": result.llava_response,
        "context_injected": result.context_injected,
        
        "detection": {
            "raw_boxes": result.dino_raw_boxes,
            "labels": result.dino_labels,
            "scores": result.dino_scores,
        },
        
        "gatekeeper": {
            "verified_boxes": result.verified_boxes,
            "rejected_boxes": result.rejected_boxes,
        },
        
        "segmentation": {
            "num_masks": len(result.masks),
            # Note: masks are not serialized (too large)
        },
        
        "errors": result.errors,
    }
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✓ Results exported to: {output_path}")
    return export_data

In [ ]:
# Export results
if 'result' in dir():
    output_path = PROJECT_ROOT / "output" / "pipeline_result.json"
    output_path.parent.mkdir(exist_ok=True)
    
    exported = export_result_to_json(result, str(output_path))
    
    print("\n📄 Exported Data Preview:")
    print(json.dumps(exported, indent=2)[:1000] + "...")

---
## ✅ Complete!

You have successfully run the TriMedAgent pipeline using the **Orchestrator Pattern**.

### Key Takeaways:

1. **Thin Client**: The Orchestrator does not load any models - all inference happens on workers
2. **Modular**: Each stage can be run independently or as a full chain
3. **Verifiable**: Gatekeeper filters false positives with transparent scoring
4. **Extensible**: Easy to add new workers or modify the pipeline

### Next Steps:
- Try different queries and images
- Adjust thresholds in `serve/labels.json`
- Add custom workers by extending `WORKER_MAP`